# Libraries

In [1]:
import os
import random
import numpy as np
from PIL import Image
from tqdm import tqdm
import pydicom
import pandas as pd
from pydicom.multival import MultiValue
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import csv
import timm
import torch.nn as nn

# optional: for AUC
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, matthews_corrcoef, roc_curve,
    brier_score_loss, balanced_accuracy_score, accuracy_score
)

In [2]:
import pandas as pd

df = pd.read_csv("/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/dhu_filtered.csv")
df['binary_label'].value_counts()

binary_label
1    1458
0    1407
Name: count, dtype: int64

In [3]:
# TRAIN_CSV = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_train.csv"
# VAL_CSV   = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_val.csv"
# TEST_CSV  = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_test.csv"

TRAIN_CSV = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_train_reduced.csv"
VAL_CSV   = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_val_reduced.csv"
TEST_CSV  = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_test_reduced.csv"

# ===============================
# Task configuration
# ===============================
# NUM_CLASSES = 4
# CLASS_NAMES = ["NORMAL", "DRUSEN", "DME", "CNV"]

NUM_CLASSES = 2
CLASS_NAMES = ["NORMAL", "DISEASE"]


# Label meaning:
# four_class_label:
# 0 -> NORMAL
# 1 -> DRUSEN
# 2 -> DME
# 3 -> CNV

# binary_label:
# 0 -> NORMAL
# 1 -> DISEASE (DRUSEN / DME / CNV)

# ===============================
# Training configuration (initial)
# ===============================
IMG_SIZE = 512        # standard for ResNet18
BATCH_SIZE = 32
NUM_WORKERS = 4
NUM_EPOCHS = 30

LR = 5e-4
WEIGHT_DECAY = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
import pandas as pd
df = pd.read_csv("neh_train.csv")
print()

In [5]:
# import pandas as pd

# tr = pd.read_csv(TRAIN_CSV)
# te = pd.read_csv(TEST_CSV)
# va = pd.read_csv(VAL_CSV)

# ROOT_DIR = "/media/bharath/DATA_8TB1/Bharath/EMBED/"       

# # tr = tr[tr['ViewPosition'] == 'MLO']
# # te = te[te['ViewPosition'] == 'MLO']

# tr['new_file_path'] = [ROOT_DIR + p for p in tr['anon_dicom_path']]
# te['new_file_path'] = [ROOT_DIR + p for p in te['anon_dicom_path']]
# va['new_file_path'] = [ROOT_DIR + p for p in va['anon_dicom_path']]

# tr.to_csv(TRAIN_CSV, index=False)
# te.to_csv(TEST_CSV, index=False)
# va.to_csv(VAL_CSV, index=False)

In [6]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
SEED = 42
set_seed(SEED)

# Dataset

In [7]:
# ===============================
# Load CSVs
# ===============================
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("Train samples:", len(train_df))
print("Val samples  :", len(val_df))
print("Test samples :", len(test_df))

# ===============================
# Check required columns
# ===============================
required_cols = [
    "new_file_path",
    "label_text",
    "three_class_label",
    "binary_label",
    "patient_id"
]

for col in required_cols:
    assert col in train_df.columns, f"Missing column in train_df: {col}"
    assert col in val_df.columns,  f"Missing column in val_df: {col}"
    assert col in test_df.columns,  f"Missing column in test_df: {col}"

print("✅ All required columns present")

# ===============================
# Class distribution
# ===============================
print("\nTrain distribution:")
print(train_df["label_text"].value_counts())

print("\nValidation distribution:")
print(val_df["label_text"].value_counts())

print("\nTest distribution:")
print(test_df["label_text"].value_counts())

# ===============================
# Peek at data
# ===============================
train_df.head()


Train samples: 1705
Val samples  : 656
Test samples : 646
✅ All required columns present

Train distribution:
label_text
Normal    962
AMD       413
DME       330
Name: count, dtype: int64

Validation distribution:
label_text
Normal    290
AMD       225
DME       141
Name: count, dtype: int64

Test distribution:
label_text
Normal    333
AMD       219
DME        94
Name: count, dtype: int64


,label_text,patient_id,new_file_path,three_class_label,binary_label,fold
0,Normal,NORMAL_1,/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD...,0,0,0
1,Normal,NORMAL_1,/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD...,0,0,0
2,Normal,NORMAL_1,/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD...,0,0,0
3,Normal,NORMAL_1,/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD...,0,0,0
4,Normal,NORMAL_1,/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD...,0,0,0


In [8]:
# # import pandas as pd
# from sklearn.model_selection import train_test_split

# # 1. Identify unique patients and their corresponding labels
# # We use the 'first' or 'mode' label for each patient to ensure stratification


# # 2. Load Dataset
# file_path = '/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered.csv'
# df = pd.read_csv(file_path)


# patient_data = df.groupby('patient_id')['binary_label'].agg(lambda x: x.mode()[0]).reset_index()

# # # 2. Split Patients: 10% (Train+Val) and 90% (Test)
# # train_val_patients, test_patients = train_test_split(
# #     patient_data, 
# #     test_size=0.90, 
# #     stratify=patient_data['binary_label'], 
# #     random_state=SEED
# # )

# train_val_patients, test_patients = train_test_split(
#     patient_data, 
#     test_size=0.50, 
#     stratify=patient_data['binary_label'], 
#     random_state=SEED
# )

# # 2. Split Patients: 15% (Train+Val) and 95% (Test)
# # train_val_patients, test_patients = train_test_split(
# #     patient_data, 
# #     test_size=0.95, 
# #     stratify=patient_data['binary_label'], 
# #     random_state=SEED
# # )

# # 3. Split the 10% Patient chunk into 8% Train and 2% Val (0.02/0.10 = 0.20)
# train_patients, val_patients = train_test_split(
#     train_val_patients, 
#     test_size=0.20, 
#     stratify=train_val_patients['binary_label'], 
#     random_state=SEED
# )

# # 4. Map the split patients back to the original full dataframe
# train_df = df[df['patient_id'].isin(train_patients['patient_id'])].copy()
# val_df   = df[df['patient_id'].isin(val_patients['patient_id'])].copy()
# test_df  = df[df['patient_id'].isin(test_patients['patient_id'])].copy()

# # --- Verification ---
# print(f"Total Patients: {len(patient_data)}")
# print(f"Train Patients: {len(train_patients)} | Val Patients: {len(val_patients)} | Test Patients: {len(test_patients)}")
# print("-" * 30)
# print(f"Total Samples: {len(df)}")
# print(f"Train Samples: {len(train_df)} ({len(train_df)/len(df):.1%})")
# print(f"Val Samples:   {len(val_df)} ({len(val_df)/len(df):.1%})")
# print(f"Test Samples:  {len(test_df)} ({len(test_df)/len(df):.1%})")

# # Check for leakage (should be 0)
# overlap = set(train_df['patient_id']).intersection(set(test_df['patient_id']))
# print(f"\nPatient ID Overlap between Train and Test: {len(overlap)}")

In [9]:
# file_path = '/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/dhu_filtered.csv'
# import pandas as pd
# df = pd.read_csv(file_path)
# # 1. Identify unique patients and their specific category (AMD, DME, or NORMAL)
# # We use the first part of the patient_id string to identify the type
# patient_data = df.groupby('patient_id').first().reset_index()
# patient_data['category'] = patient_data['patient_id'].str.extract(r'([A-Za-z]+)')

# # Define the pool of patients
# all_patients = patient_data.copy()

# # 10 percent 

# # # 2. SELECT VALIDATION PATIENTS (1 Normal, 1 from AMD/DME pool)
# # val_normal = all_patients[all_patients['category'] == 'NORMAL'].sample(1, random_state=SEED)
# # val_abnormal = all_patients[all_patients['category'].isin(['AMD'])].sample(1, random_state=SEED)
# # val_dme    = all_patients[all_patients['category'] == 'DME'].sample(1, random_state=42)

# # val_list = pd.concat([val_normal, val_abnormal, val_dme])

# # # Remove validation from pool
# # pool_after_val = all_patients[~all_patients['patient_id'].isin(val_list['patient_id'])]

# # # 3. SELECT TRAINING PATIENTS (1 Normal, 1 AMD, 1 DME)
# # # Note: Using seed 42 as requested
# # train_normal = pool_after_val[pool_after_val['category'] == 'NORMAL'].sample(1, random_state=42)
# # train_amd    = pool_after_val[pool_after_val['category'] == 'AMD'].sample(1, random_state=42)
# # train_dme    = pool_after_val[pool_after_val['category'] == 'DME'].sample(1, random_state=42)
# # train_list   = pd.concat([train_normal, train_amd, train_dme])

# # 50 percent

# # 2. SELECT VALIDATION PATIENTS (1 Normal, 1 from AMD/DME pool)
# val_normal = all_patients[all_patients['category'] == 'NORMAL'].sample(2, random_state=SEED)
# val_abnormal = all_patients[all_patients['category'].isin(['AMD'])].sample(1, random_state=SEED)
# val_dme    = all_patients[all_patients['category'] == 'DME'].sample(1, random_state=42)

# val_list = pd.concat([val_normal, val_abnormal, val_dme])

# # Remove validation from pool
# pool_after_val = all_patients[~all_patients['patient_id'].isin(val_list['patient_id'])]

# # 3. SELECT TRAINING PATIENTS (1 Normal, 1 AMD, 1 DME)
# # Note: Using seed 42 as requested
# train_normal = pool_after_val[pool_after_val['category'] == 'NORMAL'].sample(6, random_state=42)
# train_amd    = pool_after_val[pool_after_val['category'] == 'AMD'].sample(4, random_state=42)
# train_dme    = pool_after_val[pool_after_val['category'] == 'DME'].sample(5, random_state=42)
# train_list   = pd.concat([train_normal, train_amd, train_dme])



# # 4. REMAINING ARE TEST
# test_list = pool_after_val[~pool_after_val['patient_id'].isin(train_list['patient_id'])]

# # 5. MAP BACK TO DATASET
# train_df = df[df['patient_id'].isin(train_list['patient_id'])].copy()
# val_df   = df[df['patient_id'].isin(val_list['patient_id'])].copy()
# test_df  = df[df['patient_id'].isin(test_list['patient_id'])].copy()

# # --- VERIFICATION ---
# print(f"Train Patients ({len(train_list)}): {train_list['patient_id'].tolist()}")
# print(f"Val Patients   ({len(val_list)}): {val_list['patient_id'].tolist()}")
# print(f"Test Patients  ({len(test_list)})")
# print("-" * 30)
# print(f"Image Counts -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# print(f"Total Samples: {len(df)}")
# print(f"Train Samples: {len(train_df)} ({len(train_df)/len(df):.1%})")
# print(f"Val Samples:   {len(val_df)} ({len(val_df)/len(df):.1%})")
# print(f"Test Samples:  {len(test_df)} ({len(test_df)/len(df):.1%})")


In [10]:
# import pandas as pd
# from sklearn.model_selection import train_test_split
# import numpy as np
# import random
# import torch

# # 1. Set Seed for Reproducibility
# def set_seed(seed=42):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed_all(seed)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

# SEED = 42
# set_seed(SEED)

# # 2. Load Dataset
# file_path = '/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/octC8_filtered.csv'
# df = pd.read_csv(file_path)

# # 3. Split the entire data into 10% (for train+val) and 90% (for test)

# # # This ensures 90% of the total data goes to test_df
# # train_val_df, test_df = train_test_split(
# #     df, 
# #     test_size=0.90, 
# #     stratify=df['binary_label'], 
# #     random_state=SEED
# # )


# train_val_df, test_df = train_test_split(
#     df, 
#     test_size=0.50, 
#     stratify=df['binary_label'], 
#     random_state=SEED
# )


# # This ensures 95% of the total data goes to test_df
# # train_val_df, test_df = train_test_split(
# #     df, 
# #     test_size=0.95, 
# #     stratify=df['binary_label'], 
# #     random_state=SEED
# # )

# # 4. Split the 10% chunk into 8% Train and 2% Val
# # Note: 2% of total is 20% of this 10% chunk (0.02 / 0.10 = 0.20)
# train_df, val_df = train_test_split(
#     train_val_df, 
#     test_size=0.20, 
#     stratify=train_val_df['binary_label'], 
#     random_state=SEED
# )

# # Verification
# print(f"Total samples: {len(df)}")
# print(f"Train size (8%): {len(train_df)}")
# print(f"Val size   (2%): {len(val_df)}")
# print(f"Test size (90%): {len(test_df)}")

# # Check label distribution
# print("\nLabel Distribution (should be ~52.7% label 1):")
# print(f"Train: \n{train_df['binary_label'].value_counts()}")
# print(f"Val: \n{val_df['binary_label'].value_counts()}")

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load the data
file_path = '/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/dhu_filtered.csv'
df = pd.read_csv(file_path)

# 2. First Split: Separate the 95% test set from the 5% "remainder"
# Stratify ensures both sets have the same % of positive/negative cases
df_temp, test_df = train_test_split(
    df, 
    test_size=0.95, 
    stratify=df['binary_label'], 
    random_state=42
)

# 3. Second Split: Split the 5% remainder into Train (4%) and Valid (1%)
# Since 1% is 1/5 of the temp set, we use test_size=0.2 (20% of 5% = 1%)
train_df, val_df = train_test_split(
    df_temp, 
    test_size=0.20, 
    stratify=df_temp['binary_label'], 
    random_state=42
)

# 4. Print the distributions
def print_dist(name, d):
    counts = d['binary_label'].value_counts()
    percents = d['binary_label'].value_counts(normalize=True) * 100
    print(f"--- {name} ({len(d)} samples) ---")
    for val in counts.index:
        print(f"Label {val}: {counts[val]} ({percents[val]:.2f}%)")
    print()

print_dist("Train Set (4%)", train_df)
print_dist("Valid Set (1%)", val_df)
print_dist("Test Set (95%)", test_df)

--- Train Set (4%) (114 samples) ---
Label 1: 58 (50.88%)
Label 0: 56 (49.12%)

--- Valid Set (1%) (29 samples) ---
Label 1: 15 (51.72%)
Label 0: 14 (48.28%)

--- Test Set (95%) (2722 samples) ---
Label 1: 1385 (50.88%)
Label 0: 1337 (49.12%)



In [12]:
from PIL import Image, ImageFile
import torch

ImageFile.LOAD_TRUNCATED_IMAGES = True

class NEHOCTDataset(torch.utils.data.Dataset):
    def __init__(self, df, transform=None, label_col="binary_label"):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        try:
            img = Image.open(row["new_file_path"])

            # Ensure 3-channel RGB
            if img.mode != "RGB":
                img = img.convert("RGB")

        except Exception as e:
            raise RuntimeError(
                f"Failed to load image: {row['new_file_path']}"
            ) from e

        label = int(row[self.label_col])

        if self.transform:
            img = self.transform(img)

        return img, label


In [13]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [14]:
# import os
# TEST_CSV = "4096_3328_SD_imbalanced_val_15.csv"
# ROOT = "/media/miglab/DATA_20TB1/Datasets/EMBED"
# df = pd.read_csv(TEST_CSV)
# df["new_file_path"] = df["anon_dicom_path"].apply(
#     lambda x: os.path.join(ROOT, x) if pd.notna(x) else x
# )
# df.to_csv(TEST_CSV, index=False)

In [15]:
train_dataset = NEHOCTDataset(train_df, transform=train_transform, label_col='binary_label')
val_dataset   = NEHOCTDataset(val_df,   transform=test_transform, label_col='binary_label')
test_dataset  = NEHOCTDataset(test_df,  transform=test_transform, label_col='binary_label')


pinmem = True if torch.cuda.is_available() else False

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=pinmem)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pinmem)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pinmem)

In [16]:
images, _ = next(iter(train_loader))
print(images.min(), images.max(), images.mean())
print(images.shape)  # (B, C, H, W)

tensor(-2.1179) tensor(2.6400) tensor(-1.1305)
torch.Size([32, 3, 512, 512])


In [17]:
# raise Exception

In [18]:
# import matplotlib.pyplot as plt

# IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
# IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# def denormalize(img):
#     """
#     img: Tensor [3, H, W] after Normalize
#     """
#     return img * IMAGENET_STD + IMAGENET_MEAN
# def visualize_dataloader_batch(dataloader, num_images=4, title="After DataLoader"):
#     """
#     Visualize images exactly as seen by the model
#     """
#     images, labels = next(iter(dataloader))

#     num_images = min(num_images, images.size(0))

#     plt.figure(figsize=(4 * num_images, 4))
#     for i in range(num_images):
#         img = images[i].cpu()
#         img = denormalize(img)
#         img = img.clamp(0, 1)

#         plt.subplot(1, num_images, i + 1)
#         plt.imshow(img.permute(1, 2, 0))
#         plt.title(f"Label: {labels[i].item()}")
#         plt.axis("off")

#     plt.suptitle(title, fontsize=14)
#     plt.show()

# visualize_dataloader_batch(train_loader, num_images=8, title="Train Loader Output")
# visualize_dataloader_batch(val_loader,   num_images=8, title="Val Loader Output")
# visualize_dataloader_batch(test_loader,  num_images=8, title="Test Loader Output")

# Metrics

In [19]:
def compute_uncertainty_stats(y_proba: np.ndarray, y_true: np.ndarray):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_true  = np.asarray(y_true, dtype=np.int64)

    y_proba = np.nan_to_num(y_proba, nan=0.5, posinf=1.0, neginf=0.0)

    eps = 1e-8
    p1 = np.clip(y_proba, eps, 1.0 - eps)
    p0 = 1.0 - p1

    entropy = -(p0 * np.log(p0) + p1 * np.log(p1))
    confidence = np.maximum(p0, p1)
    uncertainty = 1.0 - confidence

    out = {
        "avg_entropy": float(np.mean(entropy)),
        "entropy_std": float(np.std(entropy)),
        "avg_uncertainty": float(np.mean(uncertainty)),
        "uncertainty_std": float(np.std(uncertainty)),
    }

    # ---- class-conditional statistics ----
    for cls in (0, 1):
        mask = (y_true == cls)

        if mask.any():
            out[f"entropy_class{cls}_avg"] = float(entropy[mask].mean())
            out[f"entropy_class{cls}_std"] = float(entropy[mask].std())
            out[f"uncertainty_class{cls}_avg"] = float(uncertainty[mask].mean())
            out[f"uncertainty_class{cls}_std"] = float(uncertainty[mask].std())
        else:
            out[f"entropy_class{cls}_avg"] = float("nan")
            out[f"entropy_class{cls}_std"] = float("nan")
            out[f"uncertainty_class{cls}_avg"] = float("nan")
            out[f"uncertainty_class{cls}_std"] = float("nan")

    return out

In [20]:
def compute_binary_class_weights(df):
    counts = df["binary_label"].value_counts().sort_index().values
    total = counts.sum()
    weights = total / (2.0 * counts)
    return torch.tensor(weights, dtype=torch.float)

class_weights = compute_binary_class_weights(train_df).to(DEVICE)

print("Class counts:")
print(train_df["binary_label"].value_counts().sort_index())

print("Class weights:", class_weights.cpu().numpy())

Class counts:
binary_label
0    56
1    58
Name: count, dtype: int64
Class weights: [1.0178572  0.98275864]


In [21]:
# import csv
# import os

# CSV_HEADER = [
#     "epoch",
#     "train_loss", "val_loss", "test_loss",
#     "train_acc", "val_acc", "test_acc",
#     "train_auc", "val_auc", "test_auc",
#     "val_pr_auc", "test_pr_auc",
#     "val_f1", "test_f1", "val_macro_f1", "test_macro_f1",
#     "val_precision", "val_recall", "val_npv",
#     "test_precision", "test_recall", "test_npv",
#     "val_specificity", "test_specificity",
#     "val_sens_at_spec_90", "test_sens_at_spec_90",
#     "avg_entropy", "entropy_std",
#     "avg_uncertainty", "uncertainty_std",
#     "entropy_class0_avg", "entropy_class0_std",
#     "entropy_class1_avg", "entropy_class1_std",
#     "uncertainty_class0_avg", "uncertainty_class0_std",
#     "uncertainty_class1_avg", "uncertainty_class1_std",
#     "tn", "fp", "fn", "tp", "n_samples"
# ]

# def append_metrics_to_csv(csv_path, row_dict, float_fmt="{:.6f}"):
#     """
#     Appends one epoch of metrics to CSV.
#     Floats are formatted to fixed precision (default .6f).
#     Creates file + header if missing.
#     """
#     file_exists = os.path.isfile(csv_path)

#     formatted_row = {}
#     for k, v in row_dict.items():
#         if isinstance(v, float):
#             if np.isnan(v):
#                 formatted_row[k] = np.nan
#             else:
#                 formatted_row[k] = float_fmt.format(v)
#         else:
#             formatted_row[k] = v

#     with open(csv_path, mode="a", newline="") as f:
#         writer = csv.DictWriter(f, fieldnames=CSV_HEADER)

#         if not file_exists:
#             writer.writeheader()

#         writer.writerow({k: formatted_row.get(k, np.nan) for k in CSV_HEADER})

In [22]:
import csv
import os

CSV_HEADER = [
    "epoch",
    "train_loss", "val_loss",
    "train_acc", "val_acc", 
    "train_auc", "val_auc",
    "val_pr_auc", 
    "val_f1", "val_macro_f1", 
    "val_precision", "val_recall", "val_npv",
    "val_specificity", 
    "val_sens_at_spec_90", 
    # "avg_entropy", "entropy_std",
    # "avg_uncertainty", "uncertainty_std",
    # "entropy_class0_avg", "entropy_class0_std",
    # "entropy_class1_avg", "entropy_class1_std",
    # "uncertainty_class0_avg", "uncertainty_class0_std",
    # "uncertainty_class1_avg", "uncertainty_class1_std",
    # "tn", "fp", "fn", "tp", "n_samples"
]

def append_metrics_to_csv(csv_path, row_dict, float_fmt="{:.6f}"):
    """
    Appends one epoch of metrics to CSV.
    Floats are formatted to fixed precision (default .6f).
    Creates file + header if missing.
    """
    file_exists = os.path.isfile(csv_path)

    formatted_row = {}
    for k, v in row_dict.items():
        if isinstance(v, float):
            if np.isnan(v):
                formatted_row[k] = np.nan
            else:
                formatted_row[k] = float_fmt.format(v)
        else:
            formatted_row[k] = v

    with open(csv_path, mode="a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_HEADER)

        if not file_exists:
            writer.writeheader()

        writer.writerow({k: formatted_row.get(k, np.nan) for k in CSV_HEADER})

In [23]:
def compute_binary_metrics(y_proba, y_true, threshold=0.5, target_spec=0.90):
    """
    y_proba: list or np.array of predicted probability for class 1
    y_true:  list or np.array of integer {0,1}
    threshold: float for converting proba -> label
    returns: dict of metrics including confusion matrix entries
    """
    y_proba = np.asarray(y_proba)
    y_true = np.asarray(y_true).astype(int)

    out = {}
    # ROC-AUC and PR-AUC (handle single-class edge)
    try:
        out['roc_auc'] = float(roc_auc_score(y_true, y_proba)) if len(np.unique(y_true)) > 1 else float("nan")
    except Exception:
        out['roc_auc'] = float("nan")
    try:
        out['pr_auc'] = float(average_precision_score(y_true, y_proba)) if len(np.unique(y_true)) > 1 else float("nan")
    except Exception:
        out['pr_auc'] = float("nan")

    # Binary predictions
    y_pred = (y_proba >= threshold).astype(int)

    # Standard metrics
    out['precision'] = float(precision_score(y_true, y_pred, zero_division=0, average='macro'))
    out['recall'] = float(recall_score(y_true, y_pred, zero_division=0, average='macro'))    # sensitivity
    out['f1'] = float(f1_score(y_true, y_pred, zero_division=0))
    out['macro_f1'] = float(
        f1_score(y_true, y_pred, average="macro", zero_division=0)
    )
    out['balanced_acc'] = float(balanced_accuracy_score(y_true, y_pred))
    # MCC (may raise if degenerate)
    # try:
    #     out['mcc'] = float(matthews_corrcoef(y_true, y_pred))
    # except Exception:
    #     out['mcc'] = float("nan")

    # Confusion matrix

    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
        out['tn'] = int(tn); out['fp'] = int(fp); out['fn'] = int(fn); out['tp'] = int(tp)
        out['specificity'] = float(tn / (tn + fp)) if (tn + fp) > 0 else float("nan")
        out['sensitivity'] = float(tp / (tp + fn)) if (tp + fn) > 0 else float("nan")
        out['npv'] = float(tn / (tn + fn)) if (tn + fn) > 0 else float("nan")

    except Exception:
        out['tn'] = out['fp'] = out['fn'] = out['tp'] = 0
        out['specificity'] = out['sensitivity'] = float("nan")

    # Brier score

    try:
        out['brier'] = float(brier_score_loss(y_true, y_proba))
    except Exception:
        out['brier'] = float("nan")

    out['n_samples'] = int(len(y_true))
    out['threshold'] = float(threshold)
    try:
        thresh_list = np.linspace(0, 1, 200)
        best_sens = float("nan")
        best_thr = float("nan")

        for thr in thresh_list:
            yp = (y_proba >= thr).astype(int)
            cm = confusion_matrix(y_true, yp, labels=[0,1])
            tn, fp, fn, tp = cm.ravel()
            spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
            sens = tp / (tp + fn) if (tp + fn) > 0 else float("nan")

            if not np.isnan(spec) and spec >= target_spec:
                # choose largest sensitivity among valid thresholds
                if np.isnan(best_sens) or sens > best_sens:
                    best_sens = sens
                    best_thr = thr

        out["sens_at_spec"] = best_sens
        out["thr_at_spec"] = best_thr
        out["target_spec"] = target_spec

    except Exception:
        out["sens_at_spec"] = float("nan")
        out["thr_at_spec"] = float("nan")
        out["target_spec"] = target_spec
    
    try:
        u_stats = compute_uncertainty_stats(y_proba)
        out.update(u_stats)
    except Exception:
        for k in [
            "avg_entropy","entropy_std",
            "avg_uncertainty","uncertainty_std",
            "entropy_class0_avg","entropy_class0_std",
            "entropy_class1_avg","entropy_class1_std",
            "uncertainty_class0_avg","uncertainty_class0_std",
            "uncertainty_class1_avg","uncertainty_class1_std",
        ]:
            out[k] = float("nan")
    return out

# def print_epoch_summary(epoch, train_loss, val_loss, test_loss, train_auc, val_auc, test_auc, train_acc, val_acc, test_acc, metrics, uncert):
#     print("=" * 90)
#     print(f"Epoch {epoch}")
#     print("-" * 90)
#     print(f"Train | Loss={(train_loss):.4f}  AUC={(train_auc):.4f}  Acc={(train_acc):.4f}")
#     print(f"Val   | Loss={(val_loss):.4f}  AUC={(val_auc):.4f}  Acc={(val_acc):.4f}")
#     print(f"Test  | Loss={(test_loss):.4f}  AUC={(test_auc):.4f}  Acc={(test_acc):.4f}")

#     print("\nClassification Metrics:")
#     print(
#         f"  ROC-AUC={(metrics['roc_auc']):.4f}  PR-AUC={(metrics['pr_auc']):.4f}  Brier_score={(metrics['brier']):.4f}  "
#         f"F1={(metrics['f1']):.4f}  Macro-F1={(metrics['macro_f1']):.4f}  Precision={(metrics['precision']):.4f}  Recall={(metrics['recall']):.4f}  NPV={(metrics['npv']):.4f}" f"  Specificity={(metrics['specificity']):.4f}  "  f"Sens@Spec90={(metrics['sens_at_spec']):.4f} ")

#     print("\nUncertainty Metrics:")
#     print(
#         f"  AvgEntropy={(uncert['avg_entropy']):.4f} ± {(uncert['entropy_std']):.4f} | " f"AvgUncertainty={(uncert['avg_uncertainty']):.4f} ± {(uncert['uncertainty_std']):.4f}" f"  Entropy(C0)={(uncert['entropy_class0_avg']):.4f}  " f"Entropy(C1)={(uncert['entropy_class1_avg']):.4f}" f"  Uncertainty(C0)={(uncert['uncertainty_class0_avg']):.4f}  " f"Uncertainty(C1)={(uncert['uncertainty_class1_avg']):.4f}"
#     )

#     print("\nConfusion Matrix:" f"  TP={metrics['tp']}  FP={metrics['fp']} " f"FN={metrics['fn']}  TN={metrics['tn']}")
#     print("=" * 90)

def print_epoch_summary(epoch, train_loss, val_loss, test_loss, train_auc, val_auc, test_auc, train_acc, val_acc, test_acc, metrics):
    print("=" * 90)
    print(f"Epoch {epoch}")
    print("-" * 90)
    print(f"Train | Loss={(train_loss):.4f}  AUC={(train_auc):.4f}  Acc={(train_acc):.4f}")
    print(f"Val   | Loss={(val_loss):.4f}  AUC={(val_auc):.4f}  Acc={(val_acc):.4f}")
    print(f"Test  | Loss={(test_loss):.4f}  AUC={(test_auc):.4f}  Acc={(test_acc):.4f}")

    print("\nClassification Metrics:")
    print(
        f"  ROC-AUC={(metrics['roc_auc']):.4f}  PR-AUC={(metrics['pr_auc']):.4f}  Brier_score={(metrics['brier']):.4f}  "
        f"F1={(metrics['f1']):.4f}  Macro-F1={(metrics['macro_f1']):.4f}  Precision={(metrics['precision']):.4f}  Recall={(metrics['recall']):.4f}  NPV={(metrics['npv']):.4f}" f"  Specificity={(metrics['specificity']):.4f}  "  f"Sens@Spec90={(metrics['sens_at_spec']):.4f} ")

    # print("\nUncertainty Metrics:")
    # print(
    #     f"  AvgEntropy={(uncert['avg_entropy']):.4f} ± {(uncert['entropy_std']):.4f} | " f"AvgUncertainty={(uncert['avg_uncertainty']):.4f} ± {(uncert['uncertainty_std']):.4f}" f"  Entropy(C0)={(uncert['entropy_class0_avg']):.4f}  " f"Entropy(C1)={(uncert['entropy_class1_avg']):.4f}" f"  Uncertainty(C0)={(uncert['uncertainty_class0_avg']):.4f}  " f"Uncertainty(C1)={(uncert['uncertainty_class1_avg']):.4f}"
    # )

    print("\nConfusion Matrix:" f"  TP={metrics['tp']}  FP={metrics['fp']} " f"FN={metrics['fn']}  TN={metrics['tn']}")
    print("=" * 90)

# Model

In [24]:
def get_resnet34_binary(pretrained=True, num_classes=2):
    model = models.resnet34(pretrained=pretrained)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes) 
    return model

model = get_resnet34_binary(pretrained=True, num_classes=NUM_CLASSES)
model = model.to(DEVICE)

/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [25]:
criterion = nn.CrossEntropyLoss(weight=class_weights)   # expects logits and class indices
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

# Loops

In [26]:
def set_bn_eval(m):
    if isinstance(m, nn.BatchNorm2d):
        m.eval()
        
def train(model, loader, optimizer, criterion, device):
    model.train()
    # freeze_batchnorm(model)
    # model.eval()
    model.apply(set_bn_eval)
    running_loss = 0.0
    preds_all = []
    targets_all = []
    # c=0
    for imgs, targets in tqdm(loader, desc="Train", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(imgs)                 # shape [B, 2]
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds = logits.detach().softmax(dim=1)[:,1].cpu().numpy().tolist()  # probability of class 1
        preds_all.extend(preds)
        targets_all.extend(targets.cpu().numpy().tolist())
        # if c == 10:
        #     break
        # c+=1

    epoch_loss = running_loss / len(loader.dataset)
    try:
        auc = roc_auc_score(targets_all, preds_all)
    except Exception:
        auc = float("nan")
    acc = accuracy_score(targets_all, [1 if p>=0.5 else 0 for p in preds_all])
    return epoch_loss, auc, acc

In [27]:
def validate(model, loader, criterion, device, threshold=0.5):
    model.eval()
    running_loss = 0.0
    preds_proba = []   
    targets_all = []
    # c=0
    with torch.no_grad():
        for imgs, targets in tqdm(loader, desc="Val", leave=False):
            imgs = imgs.to(device)
            targets = targets.to(device)

            logits = model(imgs)
            loss = criterion(logits, targets)
            running_loss += loss.item() * imgs.size(0)

            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds_proba.extend(probs.tolist())
            targets_all.extend(targets.cpu().numpy().tolist())
            # if c == 10:
            #     break
            # c+=1

    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_binary_metrics(preds_proba, targets_all, threshold=threshold)
    y_true = np.array(targets_all)
    y_proba = np.array(preds_proba)
    try:
        auc = roc_auc_score(y_true, y_proba)
    except Exception:
        auc = float("nan")
    acc = accuracy_score(y_true, [1 if p>=0.5 else 0 for p in y_proba])
    return epoch_loss, auc, acc, y_proba, metrics

In [28]:
def test(model, loader, criterion, device, threshold=0.5):
    model.eval()
    running_loss = 0.0
    preds_proba = []   # probability of class 1
    targets_all = []
    # c=0
    with torch.no_grad():
        for imgs, targets in tqdm(loader, desc="Test", leave=False):
            imgs = imgs.to(device)
            targets = targets.to(device)

            logits = model(imgs)
            loss = criterion(logits, targets)
            running_loss += loss.item() * imgs.size(0)

            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds_proba.extend(probs.tolist())
            targets_all.extend(targets.cpu().numpy().tolist())
            # if c == 10:
            #     break
            # c+=1
    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_binary_metrics(preds_proba, targets_all, threshold=threshold)
    y_true = np.array(targets_all)
    y_proba = np.array(preds_proba)
    try:
        auc = roc_auc_score(y_true, y_proba)
    except Exception:
        auc = float("nan")
    acc = accuracy_score(y_true, [1 if p>=0.5 else 0 for p in y_proba])
    uncert = compute_uncertainty_stats(y_proba, y_true)
    return epoch_loss, auc, acc, y_proba, metrics, uncert

# Training

In [29]:
# models_path = "ResNet34_neh_reduced_5e4"
# os.makedirs(models_path, exist_ok=True)
# log_file = os.path.join(models_path, "training_log.csv")

# BEST_MACRO_F1 = -1.0

# # checkpoint_path = "/media/miglab/DATA_20TB1/Uncertainty/ResNet34+GradAug_4096_balanced/epoch_2.pth"
# # checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

# # model.load_state_dict(checkpoint["state_dict"])
# # model.to(DEVICE)

# for epoch in range(1, NUM_EPOCHS + 1):
#     print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

#     train_loss, train_auc, train_acc = train(
#         model, train_loader, optimizer, criterion, DEVICE
#     )

#     val_loss, val_auc, val_acc, val_probs, val_metrics = validate(
#         model, val_loader, criterion, DEVICE, threshold=0.5
#     )

#     test_loss, test_auc, test_acc, test_probs, test_metrics, test_uncert = test(
#         model, test_loader, criterion, DEVICE, threshold=0.5
#     )

#     if not np.isnan(val_auc):
#         scheduler.step(val_auc)

#     ckpt = {
#         "epoch": epoch,
#         "state_dict": model.state_dict(),
#         "optimizer": optimizer.state_dict(),
#         "val_auc": val_metrics["roc_auc"]
#     }

#     torch.save(ckpt, os.path.join(models_path, f"epoch_{epoch}.pth"))

#     if val_metrics["macro_f1"] > BEST_MACRO_F1:
#         BEST_MACRO_F1 = val_metrics["macro_f1"]
#         torch.save(ckpt, os.path.join(models_path, "best_model.pth"))

#     row = {
#         "epoch": epoch,
#         "train_loss": train_loss, "val_loss": val_loss, "test_loss": test_loss,
#         "train_acc": train_acc, "val_acc": val_acc, "test_acc": test_acc,
#         "train_auc": train_auc, "val_auc": val_metrics["roc_auc"], "test_auc": test_metrics["roc_auc"],
#         "val_pr_auc": val_metrics["pr_auc"], "test_pr_auc": test_metrics["pr_auc"], 
#         "val_f1": val_metrics["f1"],  "test_f1": test_metrics["f1"], "val_macro_f1": val_metrics["macro_f1"], "test_macro_f1": test_metrics["macro_f1"],
#         "val_precision": val_metrics["precision"], "val_recall": val_metrics["recall"], "val_npv": val_metrics["npv"],
#         "test_precision": test_metrics["precision"], "test_recall": test_metrics["recall"], "test_npv": test_metrics["npv"],
#         "val_specificity": val_metrics["specificity"], "test_specificity": test_metrics["specificity"], 
#         "val_sens_at_spec_90": val_metrics['sens_at_spec'], "test_sens_at_spec_90": test_metrics['sens_at_spec'],
#         "avg_entropy": test_uncert['avg_entropy'], "entropy_std": test_uncert['entropy_std'],
#         "avg_uncertainty": test_uncert['avg_uncertainty'], "uncertainty_std": test_uncert['uncertainty_std'],
#         "entropy_class0_avg": test_uncert['entropy_class0_avg'], "entropy_class0_std": test_uncert['entropy_class0_std'], "entropy_class1_avg": test_uncert['entropy_class1_avg'], 
#         "entropy_class1_std": test_uncert['entropy_class1_std'], "uncertainty_class0_avg": test_uncert['uncertainty_class0_avg'], "uncertainty_class0_std": test_uncert['uncertainty_class0_std'], 
#         "uncertainty_class1_avg": test_uncert['uncertainty_class1_avg'], "uncertainty_class1_std": test_uncert['uncertainty_class1_std'],
#         "tn": test_metrics["tn"], "fp": test_metrics["fp"], "fn": test_metrics["fn"], "tp": test_metrics["tp"], "n_samples": test_metrics["n_samples"],
#     }

#     append_metrics_to_csv(log_file, row)

#     print_epoch_summary(
#         epoch,
#         train_loss,
#         val_loss,
#         test_loss,
#         train_auc,
#         val_auc,
#         test_auc,
#         train_acc,
#         val_acc,
#         test_acc,
#         test_metrics,
#         test_uncert
#     )

In [30]:
models_path = "Five_percent_20p_ResNet34_neh_reduced_5e4_finetuned_dhu_fc_freezed_batchNorm"
# models_path = "ResNet34_neh_reduced_5e4_finetuned_dhu_fc_freezed_batchNorm"

os.makedirs(models_path, exist_ok=True)
log_file = os.path.join(models_path, "training_log.csv")


checkpoint_path = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4/epoch_8.pth"
checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

model.load_state_dict(checkpoint["state_dict"])
model.to(DEVICE)

BEST_MACRO_F1 = -1.0

NUM_EPOCHS = 20

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(
    model.fc.parameters(),
    lr=1e-3,              # higher LR is safe here
    weight_decay=1e-4
)


# for param in model.layer4.parameters():
#     param.requires_grad = True

# optimizer = torch.optim.AdamW([
#     {"params": model.layer4.parameters(), "lr": 1e-4},
#     {"params": model.fc.parameters(),     "lr": 1e-3},
# ], weight_decay=1e-4)

# model.eval()
# for m in model.modules():
#     if isinstance(m, nn.BatchNorm2d):
#         m.eval()


# checkpoint_path = "/media/miglab/DATA_20TB1/Uncertainty/ResNet34+GradAug_4096_balanced/epoch_2.pth"
# checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

# model.load_state_dict(checkpoint["state_dict"])
# model.to(DEVICE)

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

    train_loss, train_auc, train_acc = train(
        model, train_loader, optimizer, criterion, DEVICE
    )

    val_loss, val_auc, val_acc, val_probs, val_metrics = validate(
        model, val_loader, criterion, DEVICE, threshold=0.5
    )

    # test_loss, test_auc, test_acc, test_probs, test_metrics, test_uncert = test(
    #     model, test_loader, criterion, DEVICE, threshold=0.5
    # )

    if not np.isnan(val_auc):
        scheduler.step(val_auc)

    ckpt = {
        "epoch": epoch,
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "val_auc": val_metrics["roc_auc"]
    }

    # torch.save(ckpt, os.path.join(models_path, f"epoch_{epoch}.pth"))

    if val_metrics["macro_f1"] > BEST_MACRO_F1:
        BEST_MACRO_F1 = val_metrics["macro_f1"]
        torch.save(ckpt, os.path.join(models_path, "best_model.pth"))

    row = {
        "epoch": epoch,
        "train_loss": train_loss, "val_loss": val_loss,
        "train_acc": train_acc, "val_acc": val_acc, 
        "train_auc": train_auc, "val_auc": val_metrics["roc_auc"], 
        "val_pr_auc": val_metrics["pr_auc"],
        "val_f1": val_metrics["f1"],  "val_macro_f1": val_metrics["macro_f1"], 
        "val_precision": val_metrics["precision"], "val_recall": val_metrics["recall"], "val_npv": val_metrics["npv"],
        
        "val_specificity": val_metrics["specificity"],
        "val_sens_at_spec_90": val_metrics['sens_at_spec'], 
    }

    append_metrics_to_csv(log_file, row)

    print_epoch_summary(
        epoch,
        train_loss,
        val_loss,
        val_loss,
        train_auc,
        val_auc,
        val_auc,
        train_acc,
        val_acc,
        val_acc,
        val_metrics,
    )


Epoch 1/20


Epoch 1
------------------------------------------------------------------------------------------
Train | Loss=0.4878  AUC=0.9172  Acc=0.8158
Val   | Loss=0.3725  AUC=0.9429  Acc=0.8621
Test  | Loss=0.3725  AUC=0.9429  Acc=0.8621

Classification Metrics:
  ROC-AUC=0.9429  PR-AUC=0.9650  Brier_score=0.0983  F1=0.8750  Macro-F1=0.8606  Precision=0.8701  Recall=0.8595  NPV=0.9167  Specificity=0.7857  Sens@Spec90=0.8667 

Confusion Matrix:  TP=14  FP=3 FN=1  TN=11

Epoch 2/20


Epoch 2
------------------------------------------------------------------------------------------
Train | Loss=0.3387  AUC=0.9295  Acc=0.9123
Val   | Loss=0.3551  AUC=0.9381  Acc=0.8621
Test  | Loss=0.3551  AUC=0.9381  Acc=0.8621

Classification Metrics:
  ROC-AUC=0.9381  PR-AUC=0.9616  Brier_score=0.0878  F1=0.8667  Macro-F1=0.8619  Precision=0.8619  Recall=0.8619  NPV=0.8571  Specificity=0.8571  Sens@Spec90=0.8667 

Confusion Matrix:  TP=13  FP=2 FN=2  TN=12

Epoch 3/20


Epoch 3
------------------------------------------------------------------------------------------
Train | Loss=0.3531  AUC=0.9258  Acc=0.9211
Val   | Loss=0.3429  AUC=0.9381  Acc=0.8621
Test  | Loss=0.3429  AUC=0.9381  Acc=0.8621

Classification Metrics:
  ROC-AUC=0.9381  PR-AUC=0.9616  Brier_score=0.0836  F1=0.8667  Macro-F1=0.8619  Precision=0.8619  Recall=0.8619  NPV=0.8571  Specificity=0.8571  Sens@Spec90=0.8667 

Confusion Matrix:  TP=13  FP=2 FN=2  TN=12

Epoch 4/20


Epoch 4
------------------------------------------------------------------------------------------
Train | Loss=0.3541  AUC=0.9249  Acc=0.9298
Val   | Loss=0.3276  AUC=0.9429  Acc=0.8621
Test  | Loss=0.3276  AUC=0.9429  Acc=0.8621

Classification Metrics:
  ROC-AUC=0.9429  PR-AUC=0.9650  Brier_score=0.0872  F1=0.8667  Macro-F1=0.8619  Precision=0.8619  Recall=0.8619  NPV=0.8571  Specificity=0.8571  Sens@Spec90=0.8667 

Confusion Matrix:  TP=13  FP=2 FN=2  TN=12

Epoch 5/20


...

Epoch 16
------------------------------------------------------------------------------------------
Train | Loss=0.2443  AUC=0.9369  Acc=0.9386
Val   | Loss=0.2402  AUC=0.9429  Acc=0.9310
Test  | Loss=0.2402  AUC=0.9429  Acc=0.9310

Classification Metrics:
  ROC-AUC=0.9429  PR-AUC=0.9632  Brier_score=0.0695  F1=0.9286  Macro-F1=0.9310  Precision=0.9375  Recall=0.9333  NPV=0.8750  Specificity=1.0000  Sens@Spec90=0.8667 

Confusion Matrix:  TP=13  FP=0 FN=2  TN=14

Epoch 17/20


Epoch 17
------------------------------------------------------------------------------------------
Train | Loss=0.2377  AUC=0.9449  Acc=0.9298
Val   | Loss=0.2382  AUC=0.9429  Acc=0.9310
Test  | Loss=0.2382  AUC=0.9429  Acc=0.9310

Classification Metrics:
  ROC-AUC=0.9429  PR-AUC=0.9632  Brier_score=0.0677  F1=0.9286  Macro-F1=0.9310  Precision=0.9375  Recall=0.9333  NPV=0.8750  Specificity=1.0000  Sens@Spec90=0.8667 

Confusion Matrix:  TP=13  FP=0 FN=2  TN=14

Epoch 18/20


Epoch 18
------------------------------------------------------------------------------------------
Train | Loss=0.2284  AUC=0.9480  Acc=0.9298
Val   | Loss=0.2363  AUC=0.9429  Acc=0.9310
Test  | Loss=0.2363  AUC=0.9429  Acc=0.9310

Classification Metrics:
  ROC-AUC=0.9429  PR-AUC=0.9632  Brier_score=0.0670  F1=0.9286  Macro-F1=0.9310  Precision=0.9375  Recall=0.9333  NPV=0.8750  Specificity=1.0000  Sens@Spec90=0.8667 

Confusion Matrix:  TP=13  FP=0 FN=2  TN=14

Epoch 19/20


Epoch 19
------------------------------------------------------------------------------------------
Train | Loss=0.2413  AUC=0.9375  Acc=0.9298
Val   | Loss=0.2338  AUC=0.9429  Acc=0.9310
Test  | Loss=0.2338  AUC=0.9429  Acc=0.9310

Classification Metrics:
  ROC-AUC=0.9429  PR-AUC=0.9632  Brier_score=0.0673  F1=0.9286  Macro-F1=0.9310  Precision=0.9375  Recall=0.9333  NPV=0.8750  Specificity=1.0000  Sens@Spec90=0.8667 

Confusion Matrix:  TP=13  FP=0 FN=2  TN=14

Epoch 20/20


Epoch 20
------------------------------------------------------------------------------------------
Train | Loss=0.2247  AUC=0.9480  Acc=0.9474
Val   | Loss=0.2333  AUC=0.9429  Acc=0.9310
Test  | Loss=0.2333  AUC=0.9429  Acc=0.9310

Classification Metrics:
  ROC-AUC=0.9429  PR-AUC=0.9632  Brier_score=0.0663  F1=0.9286  Macro-F1=0.9310  Precision=0.9375  Recall=0.9333  NPV=0.8750  Specificity=1.0000  Sens@Spec90=0.8667 

Confusion Matrix:  TP=13  FP=0 FN=2  TN=14


In [31]:
# raise Exception

In [32]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.isotonic import IsotonicRegression
from typing import Tuple

# =====================================================
# 1. VENN-ABERS CLASS DEFINITION
# =====================================================
class VennAbersBinary:
    def __init__(self):
        self.calib_scores = None
        self.calib_labels = None
        self._fitted = False

    def fit(self, scores: np.ndarray, labels: np.ndarray):
        self.calib_scores = np.asarray(scores, dtype=np.float64)
        self.calib_labels = np.asarray(labels, dtype=np.int32)
        self._fitted = True

    def _fit_isotonic(self, scores, labels):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(scores, labels)
        return ir

    def predict_interval(self, p: float) -> Tuple[float, float]:
        if not self._fitted:
            raise RuntimeError("Call fit() before predict_interval()")
        scores0 = np.append(self.calib_scores, p)
        labels0 = np.append(self.calib_labels, 0)
        ir0 = self._fit_isotonic(scores0, labels0)
        p0 = float(ir0.predict([p])[0])
        scores1 = np.append(self.calib_scores, p)
        labels1 = np.append(self.calib_labels, 1)
        ir1 = self._fit_isotonic(scores1, labels1)
        p1 = float(ir1.predict([p])[0])
        return min(p0, p1), max(p0, p1)

    def predict(self, p: float, method: str = "mean") -> float:
        p0, p1 = self.predict_interval(p)
        if method == "mean": return 0.5 * (p0 + p1)
        return p0 if method == "lower" else p1

    def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
        return np.array([self.predict(p, method=method) for p in probs])

# =====================================================
# 2. SETUP & PATHS
# =====================================================

CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/Five_percent_20p_ResNet34_neh_reduced_5e4_finetuned_dhu_fc_freezed_batchNorm/best_model.pth"
SAVE_DIR = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/Five_percent_20p_ResNet34_neh_reduced_5e4_finetuned_dhu_fc_freezed_batchNorm"
# CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_finetuned_ucsd_fc_freezed_batchNorm/best_model.pth"
# SAVE_DIR = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_finetuned_ucsd_fc_freezed_batchNorm"
# CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_finetuned_dhu_fc_freezed_batchNorm/best_model.pth"
# SAVE_DIR = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_finetuned_dhu_fc_freezed_batchNorm"

SAVE_PATH = os.path.join(SAVE_DIR, "venn_abers_fitted.pkl")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512


model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.to(DEVICE).eval()

val_probs, val_labels = [], []

print(f"Generating calibration scores for: {os.path.basename(CHECKPOINT_PATH)}")
with torch.no_grad():
    for imgs, labels in tqdm(val_loader):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]
        val_probs.extend(probs.cpu().numpy())
        val_labels.extend(labels.numpy())

# =====================================================
# 5. FIT AND SAVE
# =====================================================
va = VennAbersBinary()
va.fit(scores=np.array(val_probs), labels=np.array(val_labels))

os.makedirs(SAVE_DIR, exist_ok=True)
with open(SAVE_PATH, 'wb') as f:
    pickle.dump(va, f)

print(f"\n--- SUCCESS ---")
print(f"Venn-Abers model fitted on {len(val_probs)} validation samples.")
print(f"Saved to: {SAVE_PATH}")

/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Generating calibration scores for: best_model.pth


100%|██████████| 1/1 [00:00<00:00,  1.71it/s]


--- SUCCESS ---
Venn-Abers model fitted on 29 validation samples.
Saved to: /mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/Five_percent_20p_ResNet34_neh_reduced_5e4_finetuned_dhu_fc_freezed_batchNorm/venn_abers_fitted.pkl


In [33]:
from sklearn.metrics import f1_score

f1_macro = f1_score(val_labels, (np.array(val_probs) >= 0.5).astype(int), average="macro")
f1_macro


0.930952380952381

In [34]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score, fbeta_score
)
from sklearn.isotonic import IsotonicRegression
from typing import Tuple

# =====================================================
# 1. VENN-ABERS CLASS DEFINITION
# =====================================================
class VennAbersBinary:
    def __init__(self):
        self.calib_scores = None
        self.calib_labels = None
        self._fitted = False

    def fit(self, scores: np.ndarray, labels: np.ndarray):
        self.calib_scores = np.asarray(scores, dtype=np.float64)
        self.calib_labels = np.asarray(labels, dtype=np.int32)
        self._fitted = True

    def _fit_isotonic(self, scores, labels):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(scores, labels)
        return ir

    def predict_interval(self, p: float) -> Tuple[float, float]:
        if not self._fitted:
            raise RuntimeError("Call fit() before predict_interval()")
        scores0 = np.append(self.calib_scores, p)
        labels0 = np.append(self.calib_labels, 0)
        ir0 = self._fit_isotonic(scores0, labels0)
        p0 = float(ir0.predict([p])[0])
        scores1 = np.append(self.calib_scores, p)
        labels1 = np.append(self.calib_labels, 1)
        ir1 = self._fit_isotonic(scores1, labels1)
        p1 = float(ir1.predict([p])[0])
        return min(p0, p1), max(p0, p1)

    def predict(self, p: float, method: str = "mean") -> float:
        p0, p1 = self.predict_interval(p)
        if method == "mean": return 0.5 * (p0 + p1)
        elif method == "lower": return p0
        elif method == "upper": return p1
        else: raise ValueError("method must be 'mean', 'lower', or 'upper'")

    def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
        probs = np.asarray(probs, dtype=np.float64)
        return np.array([self.predict(p, method=method) for p in probs])

# =====================================================
# 2. SETUP & PATHS
# =====================================================

# CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_finetuned_ucsd_fc_freezed_batchNorm/best_model.pth"
# VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_finetuned_ucsd_fc_freezed_batchNorm/venn_abers_fitted.pkl"

CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/Five_percent_20p_ResNet34_neh_reduced_5e4_finetuned_dhu_fc_freezed_batchNorm/best_model.pth"
VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/Five_percent_20p_ResNet34_neh_reduced_5e4_finetuned_dhu_fc_freezed_batchNorm/venn_abers_fitted.pkl"

# CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_finetuned_dhu_fc_freezed_batchNorm/best_model.pth"
# VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_finetuned_dhu_fc_freezed_batchNorm/venn_abers_fitted.pkl"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512

# =====================================================
# 3. HELPER METRICS FUNCTION
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

# def expected_calibration_error(y_true, y_proba, n_bins=10):
#     y_true = np.asarray(y_true, dtype=int)
#     y_proba = np.asarray(y_proba, dtype=np.float64)
#     y_pred = (y_proba >= 0.5).astype(int)
#     confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
#     bins = np.linspace(0.0, 1.0, n_bins + 1)
#     bin_ids = np.digitize(confidence, bins) - 1
#     ece = 0.0
#     for b in range(n_bins):
#         mask = bin_ids == b
#         if not np.any(mask): continue
#         acc_b = np.mean(y_pred[mask] == y_true[mask])
#         conf_b = np.mean(confidence[mask])
#         ece += np.abs(acc_b - conf_b) * np.mean(mask)
#     return float(ece)


def expected_calibration_error(y_true, y_proba, n_bins=15):
    """
    Standard ECE for binary classification (Guo et al., 2017)
    """
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    # Prediction and confidence
    y_pred = (y_proba >= 0.5).astype(int)
    conf = np.maximum(y_proba, 1.0 - y_proba)

    # Bin edges: confidence ∈ [0.5, 1.0]
    bin_edges = np.linspace(0.5, 1.0, n_bins + 1)

    ece = 0.0
    n = len(y_true)

    for i in range(n_bins):
        bin_lower = bin_edges[i]
        bin_upper = bin_edges[i + 1]

        mask = (conf > bin_lower) & (conf <= bin_upper)
        if not np.any(mask):
            continue

        acc_bin = np.mean(y_pred[mask] == y_true[mask])
        conf_bin = np.mean(conf[mask])
        ece += (np.sum(mask) / n) * abs(acc_bin - conf_bin)

    return float(ece)

# =====================================================
# 5. LOAD MODEL & VENN-ABERS
# =====================================================
model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.to(DEVICE).eval()

with open(VA_PICKLE_PATH, 'rb') as f:
    va = pickle.load(f)

# =====================================================
# 6. INFERENCE
# =====================================================
# =====================================================
# 1. INFERENCE (Using existing model & test_loader)
# =====================================================

probs_all, labels_all = [], []
model.eval()

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="Testing"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        # Assuming binary classification where index 1 is the positive class
        probs = F.softmax(logits, dim=1)[:, 1]
        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels.numpy())

y_true = np.array(labels_all)
y_prob_raw = np.array(probs_all)

# =====================================================
# 2. APPLY VENN-ABERS CALIBRATION
# =====================================================
print("\nApplying Venn-Abers Calibration...")
# Using the loaded 'va' object from your previous cells
y_prob_va = va.predict_batch(y_prob_raw, method="mean")

# =====================================================
# 3. METRIC CALCULATION (In Specific Order)
# =====================================================
def get_ordered_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    conf_stats = set_size_and_singleton(y_prob, tau=0.9)
    
    # Mapping metrics to the specific requested order
    return {
        "F1 Macro": f1_score(y_true, y_pred, average="macro"),
        "F2 Macro": fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0),
        "F2 Weighted": fbeta_score(y_true, y_pred, beta=2.0, average="weighted", zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "Specificity": specificity_score(y_true, y_pred),
        "Precision Macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall Macro": recall_score(y_true, y_pred, average="macro"),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Sensitivity": recall_score(y_true, y_pred), # Sensitivity is the same as Recall
        "NPV": npv_score(y_true, y_pred),
        "Kappa": cohen_kappa_score(y_true, y_pred),
        "ECE": expected_calibration_error(y_true, y_prob),
        "Set Size": conf_stats["avg_set_size"],
        "Singleton": conf_stats["singleton_rate"]
    }

metrics_raw = get_ordered_metrics(y_true, y_prob_raw)
metrics_va = get_ordered_metrics(y_true, y_prob_va)

# =====================================================
# 4. PRINT RESULTS
# =====================================================
print("\n" + "=" * 75)
print(f"EVALUATION RESULTS")
print("=" * 75)
print(f"{'Metric':<30} | {'Before VA':<15} | {'After VA':<15}")
print("-" * 75)

# Iterate through the dictionary keys to maintain the requested order
for key in metrics_raw.keys():
    suffix = "%" if key == "Singleton" else ""
    print(f"{key:<30} | {metrics_raw[key]:<15.4f}{suffix} | {metrics_va[key]:<15.4f}{suffix}")

print("=" * 75)

/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Testing: 100%|██████████| 86/86 [00:06<00:00, 13.57it/s]



Applying Venn-Abers Calibration...

EVALUATION RESULTS
Metric                         | Before VA       | After VA       
---------------------------------------------------------------------------
F1 Macro                       | 0.9328          | 0.9317         
F2 Macro                       | 0.9329          | 0.9318         
F2 Weighted                    | 0.9326          | 0.9315         
Accuracy                       | 0.9328          | 0.9317         
AUC                            | 0.9704          | 0.9670         
Specificity                    | 0.9589          | 0.9544         
Precision Macro                | 0.9336          | 0.9323         
Recall Macro                   | 0.9332          | 0.9321         
Precision                      | 0.9581          | 0.9538         
Sensitivity                    | 0.9076          | 0.9097         
NPV                            | 0.9092          | 0.9108         
Kappa                          | 0.8656          | 0.8634       

# 50 percent results

In [35]:
# # UCSD
# Applying Venn-Abers Calibration...

# ===========================================================================
# EVALUATION RESULTS
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# F1 Macro                       | 0.9491          | 0.9487         
# F2 Macro                       | 0.9495          | 0.9487         
# F2 Weighted                    | 0.9492          | 0.9488         
# Accuracy                       | 0.9492          | 0.9488         
# AUC                            | 0.9864          | 0.9862         
# Specificity                    | 0.9604          | 0.9470         
# Precision Macro                | 0.9488          | 0.9486         
# Recall Macro                   | 0.9499          | 0.9487         
# Precision                      | 0.9638          | 0.9526         
# Sensitivity                    | 0.9393          | 0.9505         
# NPV                            | 0.9338          | 0.9446         
# Kappa                          | 0.8983          | 0.8973         
# ECE                            | 0.0146          | 0.0036         
# Set Size                       | 1.1996          | 1.1236         
# Singleton                      | 80.0369        % | 87.6448        %
# ===========================================================================

In [36]:
# # DHU 

# Applying Venn-Abers Calibration...

# ===========================================================================
# EVALUATION RESULTS
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# F1 Macro                       | 0.9304          | 0.9304         
# F2 Macro                       | 0.9316          | 0.9316         
# F2 Weighted                    | 0.9301          | 0.9301         
# Accuracy                       | 0.9304          | 0.9304         
# AUC                            | 0.9789          | 0.9789         
# Specificity                    | 0.9705          | 0.9705         
# Precision Macro                | 0.9309          | 0.9309         
# Recall Macro                   | 0.9332          | 0.9332         
# Precision                      | 0.9724          | 0.9724         
# Sensitivity                    | 0.8958          | 0.8958         
# NPV                            | 0.8893          | 0.8893         
# Kappa                          | 0.8610          | 0.8610         
# ECE                            | 0.0464          | 0.0503         
# Set Size                       | 1.3902          | 1.0962         
# Singleton                      | 60.9823        % | 90.3820        %
# ===========================================================================

In [37]:
# oct c8

# Applying Venn-Abers Calibration...

# ===========================================================================
# EVALUATION RESULTS
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# F1 Macro                       | 0.8961          | 0.8971         
# F2 Macro                       | 0.8962          | 0.8968         
# F2 Weighted                    | 0.8963          | 0.8973         
# Accuracy                       | 0.8963          | 0.8974         
# AUC                            | 0.9661          | 0.9659         
# Specificity                    | 0.8952          | 0.8828         
# Precision Macro                | 0.8960          | 0.8975         
# Recall Macro                   | 0.8963          | 0.8967         
# Precision                      | 0.9040          | 0.8952         
# Sensitivity                    | 0.8973          | 0.9107         
# NPV                            | 0.8881          | 0.8999         
# Kappa                          | 0.7923          | 0.7941         
# ECE                            | 0.0232          | 0.0293         
# Set Size                       | 1.3721          | 1.3714         
# Singleton                      | 62.7923        % | 62.8621        %
# ===========================================================================

# 10 percent results 20 epochs

In [38]:
# # UCSD
# Applying Venn-Abers Calibration...

# ===========================================================================
# EVALUATION RESULTS
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# F1 Macro                       | 0.9344          | 0.9348         
# F2 Macro                       | 0.9348          | 0.9347         
# F2 Weighted                    | 0.9343          | 0.9349         
# Accuracy                       | 0.9344          | 0.9350         
# AUC                            | 0.9801          | 0.9799         
# Specificity                    | 0.9539          | 0.9272         
# Precision Macro                | 0.9344          | 0.9350         
# Recall Macro                   | 0.9353          | 0.9346         
# Precision                      | 0.9560          | 0.9339         
# Sensitivity                    | 0.9166          | 0.9421         
# NPV                            | 0.9129          | 0.9362         
# Kappa                          | 0.8689          | 0.8696         
# ECE                            | 0.0058          | 0.0140         
# Set Size                       | 1.2093          | 1.2288         
# Singleton                      | 79.0746        % | 77.1206        %
# ===========================================================================

In [39]:
# # DHU
# Applying Venn-Abers Calibration...

# ===========================================================================
# EVALUATION RESULTS
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# F1 Macro                       | 0.9180          | 0.9105         
# F2 Macro                       | 0.9169          | 0.9103         
# F2 Weighted                    | 0.9173          | 0.9104         
# Accuracy                       | 0.9184          | 0.9105         
# AUC                            | 0.9605          | 0.9602         
# Specificity                    | 0.9835          | 0.9299         
# Precision Macro                | 0.9260          | 0.9112         
# Recall Macro                   | 0.9178          | 0.9103         
# Precision                      | 0.9807          | 0.9258         
# Sensitivity                    | 0.8521          | 0.8908         
# NPV                            | 0.8714          | 0.8967         
# Kappa                          | 0.8367          | 0.8210         
# ECE                            | 0.0452          | 0.0453         
# Set Size                       | 1.1294          | 1.0720         
# Singleton                      | 87.0578        % | 92.8007        %
# ===========================================================================

In [40]:
# # OCT c8
# Applying Venn-Abers Calibration...

# ===========================================================================
# EVALUATION RESULTS
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# F1 Macro                       | 0.8921          | 0.8846         
# F2 Macro                       | 0.8926          | 0.8843         
# F2 Weighted                    | 0.8921          | 0.8850         
# Accuracy                       | 0.8922          | 0.8850         
# AUC                            | 0.9572          | 0.9554         
# Specificity                    | 0.9145          | 0.8673         
# Precision Macro                | 0.8923          | 0.8852         
# Recall Macro                   | 0.8932          | 0.8842         
# Precision                      | 0.9181          | 0.8818         
# Sensitivity                    | 0.8719          | 0.9011         
# NPV                            | 0.8666          | 0.8887         
# Kappa                          | 0.7844          | 0.7693         
# ECE                            | 0.0267          | 0.0201         
# Set Size                       | 1.4299          | 1.3444         
# Singleton                      | 57.0099        % | 65.5614        %
# ===========================================================================

# 5 percent Results

In [41]:
# # UCSD
# Applying Venn-Abers Calibration...

# ===========================================================================
# EVALUATION RESULTS
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# F1 Macro                       | 0.9275          | 0.9277         
# F2 Macro                       | 0.9281          | 0.9274         
# F2 Weighted                    | 0.9275          | 0.9279         
# Accuracy                       | 0.9276          | 0.9279         
# AUC                            | 0.9774          | 0.9771         
# Specificity                    | 0.9500          | 0.9136         
# Precision Macro                | 0.9276          | 0.9283         
# Recall Macro                   | 0.9287          | 0.9273         
# Precision                      | 0.9526          | 0.9234         
# Sensitivity                    | 0.9073          | 0.9409         
# NPV                            | 0.9025          | 0.9331         
# Kappa                          | 0.8552          | 0.8554         
# ECE                            | 0.0081          | 0.0176         
# Set Size                       | 1.2472          | 1.2039         
# Singleton                      | 75.2794        % | 79.6089        %
# ===========================================================================

In [ ]:
# # # DHU

# Applying Venn-Abers Calibration...

# ===========================================================================
# EVALUATION RESULTS
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# F1 Macro                       | 0.9328          | 0.9317         
# F2 Macro                       | 0.9329          | 0.9318         
# F2 Weighted                    | 0.9326          | 0.9315         
# Accuracy                       | 0.9328          | 0.9317         
# AUC                            | 0.9704          | 0.9670         
# Specificity                    | 0.9589          | 0.9544         
# Precision Macro                | 0.9336          | 0.9323         
# Recall Macro                   | 0.9332          | 0.9321         
# Precision                      | 0.9581          | 0.9538         
# Sensitivity                    | 0.9076          | 0.9097         
# NPV                            | 0.9092          | 0.9108         
# Kappa                          | 0.8656          | 0.8634         
# ECE                            | 0.0535          | 0.1085         
# Set Size                       | 1.4071          | 1.6087         
# Singleton                      | 59.2946        % | 39.1256        %
# ===========================================================================

In [43]:
# # OCT c8
# Applying Venn-Abers Calibration...

# ===========================================================================
# EVALUATION RESULTS
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# F1 Macro                       | 0.8800          | 0.8812         
# F2 Macro                       | 0.8801          | 0.8816         
# F2 Weighted                    | 0.8802          | 0.8813         
# Accuracy                       | 0.8802          | 0.8813         
# AUC                            | 0.9481          | 0.9446         
# Specificity                    | 0.8781          | 0.8955         
# Precision Macro                | 0.8799          | 0.8811         
# Recall Macro                   | 0.8801          | 0.8820         
# Precision                      | 0.8883          | 0.9013         
# Sensitivity                    | 0.8821          | 0.8684         
# NPV                            | 0.8714          | 0.8610         
# Kappa                          | 0.7600          | 0.7625         
# ECE                            | 0.0330          | 0.0337         
# Set Size                       | 1.5034          | 1.4415         
# Singleton                      | 49.6601        % | 55.8516        %
# ===========================================================================

# 10 percent Results

In [44]:
# OCT c8

# Applying Venn-Abers Calibration...

# ===========================================================================
# EVALUATION RESULTS
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# F1 Macro                       | 0.8733          | 0.8749         
# F2 Macro                       | 0.8734          | 0.8748         
# F2 Weighted                    | 0.8736          | 0.8753         
# Accuracy                       | 0.8736          | 0.8753         
# AUC                            | 0.9452          | 0.9443         
# Specificity                    | 0.8706          | 0.8608         
# Precision Macro                | 0.8732          | 0.8753         
# Recall Macro                   | 0.8734          | 0.8747         
# Precision                      | 0.8815          | 0.8752         
# Sensitivity                    | 0.8763          | 0.8885         
# NPV                            | 0.8649          | 0.8754         
# Kappa                          | 0.7467          | 0.7499         
# ECE                            | 0.0445          | 0.0189         
# Set Size                       | 1.5662          | 1.3673         
# Singleton                      | 43.3779        % | 63.2732        %
# ===========================================================================

In [45]:
# dhu

# Applying Venn-Abers Calibration...

# ===========================================================================
# EVALUATION RESULTS
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# F1 Macro                       | 0.9180          | 0.9105         
# F2 Macro                       | 0.9169          | 0.9103         
# F2 Weighted                    | 0.9173          | 0.9104         
# Accuracy                       | 0.9184          | 0.9105         
# AUC                            | 0.9605          | 0.9602         
# Specificity                    | 0.9835          | 0.9299         
# Precision Macro                | 0.9260          | 0.9112         
# Recall Macro                   | 0.9178          | 0.9103         
# Precision                      | 0.9807          | 0.9258         
# Sensitivity                    | 0.8521          | 0.8908         
# NPV                            | 0.8714          | 0.8967         
# Kappa                          | 0.8367          | 0.8210         
# ECE                            | 0.0437          | 0.0482         
# Set Size                       | 1.1294          | 1.0720         
# Singleton                      | 87.0578        % | 92.8007        %
# ===========================================================================

In [46]:
# # UCSD
# Applying Venn-Abers Calibration...

# ===========================================================================
# EVALUATION RESULTS
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# F1 Macro                       | 0.9339          | 0.9335         
# F2 Macro                       | 0.9339          | 0.9333         
# F2 Weighted                    | 0.9341          | 0.9336         
# Accuracy                       | 0.9341          | 0.9337         
# AUC                            | 0.9795          | 0.9791         
# Specificity                    | 0.9284          | 0.9235         
# Precision Macro                | 0.9340          | 0.9338         
# Recall Macro                   | 0.9338          | 0.9332         
# Precision                      | 0.9347          | 0.9308         
# Sensitivity                    | 0.9393          | 0.9429         
# NPV                            | 0.9334          | 0.9368         
# Kappa                          | 0.8678          | 0.8670         
# ECE                            | 0.0195          | 0.0055         
# Set Size                       | 1.2798          | 1.2471         
# Singleton                      | 72.0216        % | 75.2869        %
# ===========================================================================


In [47]:
# import os
# import torch
# import numpy as np
# import pandas as pd
# import pickle
# from PIL import Image
# from tqdm import tqdm
# import torch.nn as nn
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader
# from torchvision import models, transforms
# from sklearn.isotonic import IsotonicRegression
# from typing import Tuple

# # =====================================================
# # 1. VENN-ABERS CLASS DEFINITION
# # =====================================================
# class VennAbersBinary:
#     def __init__(self):
#         self.calib_scores = None
#         self.calib_labels = None
#         self._fitted = False

#     def fit(self, scores: np.ndarray, labels: np.ndarray):
#         self.calib_scores = np.asarray(scores, dtype=np.float64)
#         self.calib_labels = np.asarray(labels, dtype=np.int32)
#         self._fitted = True

#     def _fit_isotonic(self, scores, labels):
#         ir = IsotonicRegression(out_of_bounds="clip")
#         ir.fit(scores, labels)
#         return ir

#     def predict_interval(self, p: float) -> Tuple[float, float]:
#         if not self._fitted:
#             raise RuntimeError("Call fit() before predict_interval()")
#         scores0 = np.append(self.calib_scores, p)
#         labels0 = np.append(self.calib_labels, 0)
#         ir0 = self._fit_isotonic(scores0, labels0)
#         p0 = float(ir0.predict([p])[0])
#         scores1 = np.append(self.calib_scores, p)
#         labels1 = np.append(self.calib_labels, 1)
#         ir1 = self._fit_isotonic(scores1, labels1)
#         p1 = float(ir1.predict([p])[0])
#         return min(p0, p1), max(p0, p1)

#     def predict(self, p: float, method: str = "mean") -> float:
#         p0, p1 = self.predict_interval(p)
#         if method == "mean": return 0.5 * (p0 + p1)
#         return p0 if method == "lower" else p1

#     def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
#         return np.array([self.predict(p, method=method) for p in probs])

# # =====================================================
# # 2. SETUP & PATHS
# # =====================================================
# CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4/epoch_8.pth"
# VAL_CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_val_reduced.csv"
# SAVE_DIR = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4"
# SAVE_PATH = os.path.join(SAVE_DIR, "venn_abers_fitted.pkl")

# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# BATCH_SIZE = 32
# IMG_SIZE = 512

# # =====================================================
# # 3. DATASET & LOADER
# # =====================================================
# test_transform = transforms.Compose([
#     transforms.Resize((IMG_SIZE, IMG_SIZE)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

# class UCSDDataset(Dataset):
#     def __init__(self, csv_path, transform):
#         self.df = pd.read_csv(csv_path)
#         self.transform = transform
#     def __len__(self): return len(self.df)
#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         img = Image.open(row["new_file_path"]).convert("RGB")
#         label = int(row["binary_label"])
#         return self.transform(img), label

# val_loader = DataLoader(UCSDDataset(VAL_CSV_PATH, test_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# # =====================================================
# # 4. EXTRACT SCORES
# # =====================================================
# model = models.resnet34(pretrained=False)
# model.fc = nn.Linear(model.fc.in_features, 2)
# checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
# model.load_state_dict(checkpoint["state_dict"])
# model.to(DEVICE).eval()

# val_probs, val_labels = [], []

# print(f"Generating calibration scores for: {os.path.basename(CHECKPOINT_PATH)}")
# with torch.no_grad():
#     for imgs, labels in tqdm(val_loader):
#         imgs = imgs.to(DEVICE)
#         logits = model(imgs)
#         probs = F.softmax(logits, dim=1)[:, 1]
#         val_probs.extend(probs.cpu().numpy())
#         val_labels.extend(labels.numpy())

# # =====================================================
# # 5. FIT AND SAVE
# # =====================================================
# va = VennAbersBinary()
# va.fit(scores=np.array(val_probs), labels=np.array(val_labels))

# os.makedirs(SAVE_DIR, exist_ok=True)
# with open(SAVE_PATH, 'wb') as f:
#     pickle.dump(va, f)

# print(f"\n--- SUCCESS ---")
# print(f"Venn-Abers model fitted on {len(val_probs)} validation samples.")
# print(f"Saved to: {SAVE_PATH}")

In [48]:
raise Exception

Exception: 

# Plots